In [ ]:
from keras.models import Sequential, Model
from keras.layers import Conv2D, Flatten, Dense, MaxPooling2D
from keras.constraints import maxnorm

In [ ]:
def build_model6(nst, nt, nc):
    model = Sequential()
    
    # Conv and Max-Pooling1 ____________________________________________________
    model.add(Conv2D(12,(1,3),activation='relu',input_shape=(nst, nt, nc)))
    model.add(MaxPooling2D((1,2)))
    
    # Conv and Max-Pooling2 ____________________________________________________
    model.add(Conv2D(24,(1,3),activation='relu',padding='same'))
    model.add(Conv2D(32,(1,3),activation='relu',padding='same'))
    model.add(MaxPooling2D((1,2)))
    
    # Conv and Max-Pooling3 ____________________________________________________
    model.add(Conv2D(64,(1,3),activation='relu',padding='same'))
    model.add(Conv2D(128,(1,3),activation='relu',padding='same'))
    model.add(MaxPooling2D((1,2)))
    
    # Conv ____________________________________________________
    model.add(Conv2D(256,(1,3),activation='relu'))
    
    # Flatten ____________________________________________________
    model.add(Flatten())
    
     # 128 Neurons ____________________________________________________
    model.add(Dense(128, activation='relu',kernel_initializer="normal",kernel_constraint=MaxNorm(3)))    
     # 32 Neurons ____________________________________________________
    model.add(Dense(32, activation='relu',kernel_initializer="normal",kernel_constraint=MaxNorm(3)))
     # 1 Neurons ____________________________________________________
    model.add(Dense(1))
    
    return model

In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import pickle
import os
import shutil
import logging
import tensorflow as tf
import keras
from GNSS_Magnitude_V1.model import build_model6


# Tracking learning rate
def get_lr_metric(optimizer):
    def lr(y_true,y_esti):
        return optimizer.learning_rate
    return lr

# Call checkpoint - Early stopping ****************
def trainCallback(checkpoint_filepath):
    
    model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_filepath,
        save_weights_only=True,
        monitor='val_loss',
        mode='min',
        save_best_only=True)

    model_early_stopping_callback = keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        min_delta=0, 
        patience=20, 
        verbose=0,
        mode='min', 
        baseline=None, 
        restore_best_weights=False)
    
    return model_checkpoint_callback,model_early_stopping_callback


def trainer(nst,nt,nc,dirout,dir_case,x1,y1,x2,y2):
    tf.random.set_seed(2)
    
    # Paths ****************************************************************
    if os.path.exists(dirout+dir_case+'/'):
        logging.warning('The folder '+dirout+dir_case+' already exist! It will be removed and made again.\n')
        shutil.rmtree(dirout+dir_case+'/')
    os.mkdir(dirout+dir_case+'/')

    checkpoint_filepath = dirout+dir_case+'/cp.weights.h5'
    hist_filepath = dirout+dir_case+'/history.p'
    model_path = dirout+dir_case+'/model.keras'
    
    # Parameters ****************************************************************
    batch_size = 128
    epochs = 200
    
    logging.info('Total epochs = %i',epochs)
    logging.info('Batch size = %i', batch_size)
 
    # Build Model ****************************************************************
    model=build_model6(nst,nt,nc)
    
    # Learning rate & Optimizer ****************************************************************
    logging.info("\n Using 'standard' learning rate decay...")
    initial_learning_rate = 1e-2
    logging.info('initial learning rate: %5.4f', initial_learning_rate)

    lr_schedule = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate,
        decay_steps=1000,
        decay_rate=0.9,
        staircase=True)

    opt = keras.optimizers.Adam(learning_rate=lr_schedule)
    lr_metric = get_lr_metric(opt)

    # Compile ****************************************************************
    model.compile(loss='mse', optimizer=opt, metrics=['mae',lr_metric])
    
    #Checkpoint ****************************************************************
    model_checkpoint_callback,model_early_stopping_callback=trainCallback(checkpoint_filepath)
    
    # Train ****************************************************************
    hist = model.fit(x1, y1,validation_data=(x2, y2),epochs=epochs,
                       batch_size=batch_size,verbose=2,
                       callbacks=[model_checkpoint_callback, model_early_stopping_callback],
                       shuffle=True)
          
    # Evaluate ****************************************************************
    loss, mae, lr = model.evaluate(x2, y2, verbose=2)
    logging.info('\nValidation:')
    logging.info('loss: %5.5f', loss)
    logging.info('mae: %5.5f', mae)
    logging.info('lr: %5.5f', lr)
 
    # Save validation info in file    
    np.savetxt(dirout+dir_case+'/Validation_values.txt',(loss,mae,lr),
           fmt='%5.5f',header='loss, mae, lr',
           delimiter=' ')
    
    # Save Model ****************************************************************
    model.load_weights(checkpoint_filepath)
    model.save(model_path)

    hist_h = hist.history
    
    logging.info('History attributes:')
    logging.info(hist_h.keys())

    with open(hist_filepath, 'wb') as file_pi:
        pickle.dump(hist_h, file_pi)
        
    # Model summary    ****************************************************************
    with open(dirout+dir_case+'/report_model.txt','w', encoding='utf-8') as fh:
        model.summary(print_fn=lambda x: fh.write(x + '\n'))
        
    dot_img_file = dirout+dir_case+'/model.pdf'
    keras.utils.plot_model(model, to_file=dot_img_file, show_shapes=True)
    
    return model

In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import os
import shutil
import logging
import statistics as statis

# Prediction & Errors **************************************************************
def predict(x_test,y_test,model,dirout,dir_case):
    pred = model.predict(x_test)
    y_pred = pred.reshape(len(y_test),)
    yr_pred = np.array([round(valy,1) for valy in y_pred])#round to 1 decimal

    # Statistics value prediction
    pred_error = np.array([round(valerr,1) for valerr in (yr_pred - y_test)])#round to 1 decimal
    abs_pred_error = np.array([round(valea,1) for valea in np.abs(yr_pred - y_test)])#round to 1 decimal
    mean_error = np.mean(abs_pred_error)
    min_error = np.min(abs_pred_error)
    max_error = np.max(abs_pred_error)
    std_error = np.std(pred_error)
    mode_error = statis.mode(abs_pred_error)
    rms_error = np.sqrt(((pred_error)**2).mean())
    
    logging.info('\nPrediction report:')
    logging.info('Mean Absolute Error: %10.3f', round(mean_error,3))
    logging.info('Min Absolute Error: %10.3f', min_error)
    logging.info('Max Absolute Error: %10.3f', max_error)
    logging.info('Std Dev: %10.3f', round(std_error,3))
    logging.info('Mode Absolute Error: %10.3f', mode_error)
    logging.info('RMSE: %10.3f', round(rms_error,3))

    # Save files with prediction results       
    if os.path.exists(dirout+dir_case):
        logging.warning('The folder '+dirout+dir_case+' already exist! It will be removed and made again.\n')
        shutil.rmtree(dirout+dir_case)
    os.mkdir(dirout+dir_case)
    
    f=open(dirout+dir_case+'/Results_Magnitude.dat','w')
    f.write('Magnitude, Predicted Mag\n')
    for ix in range(len(yr_pred)):
        f.write(str(y_test[ix])+' '+ str(yr_pred[ix])+'\n')
    f.close()

    np.savetxt(dirout+dir_case+'/Predict_Eval.txt',(mean_error,min_error,max_error,std_error,
                                                           mode_error,rms_error),
               fmt='%10.5f',header='mean_error, min_error, max_error, std_error, mode_error, rms_error',
               delimiter=' ')